# Week 5 Exercise: Company Specialized Contracts Q&A System

Build a Retrieval-Augmented Generation (RAG) assistant that answers questions about company contracts.

This notebook uses:
- Contract files from `week5/knowledge-base/contracts/`
- OpenAI embeddings (`text-embedding-3-large`)
- Chroma vector database (persistent local storage)
- Chat history in retrieval and generation
- Gradio chat UI

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages

import gradio as gr

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set. Add it to your environment or .env file.")

# Paths are relative to this notebook location in week5/
BASE_DIR = Path.cwd()
CONTRACTS_DIR = BASE_DIR / "knowledge-base" / "contracts"
VECTOR_DB_DIR = BASE_DIR / "vector_db_contracts"

MODEL_NAME = "gpt-4.1-nano"
EMBEDDING_MODEL = "text-embedding-3-large"
RETRIEVAL_K = 6

In [ ]:
def load_contract_documents() -> list[Document]:
    """Read markdown contracts directly to avoid loader version conflicts."""
    docs: list[Document] = []
    for file_path in sorted(CONTRACTS_DIR.glob("*.md")):
        text = file_path.read_text(encoding="utf-8")
        docs.append(
            Document(
                page_content=text,
                metadata={"source": file_path.name},
            )
        )
    return docs


def split_text(text: str, chunk_size: int = 1200, chunk_overlap: int = 200) -> list[str]:
    """Simple recursive-like character splitter without external dependencies."""
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    # Try coarse boundaries first to keep semantic sections together.
    sections = text.split("\n## ")
    normalized_sections = []
    for i, section in enumerate(sections):
        if i == 0:
            normalized_sections.append(section)
        else:
            normalized_sections.append("## " + section)

    chunks: list[str] = []
    for section in normalized_sections:
        start = 0
        while start < len(section):
            end = min(start + chunk_size, len(section))
            chunk = section[start:end].strip()
            if chunk:
                chunks.append(chunk)
            if end >= len(section):
                break
            start = end - chunk_overlap

    return chunks


def chunk_documents(docs: list[Document], chunk_size: int = 1200, chunk_overlap: int = 200) -> list[Document]:
    """Convert full contract docs into chunked Document objects."""
    chunked_docs: list[Document] = []
    for doc in docs:
        pieces = split_text(doc.page_content, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        for idx, piece in enumerate(pieces):
            meta = dict(doc.metadata)
            meta["chunk_index"] = idx
            chunked_docs.append(Document(page_content=piece, metadata=meta))
    return chunked_docs


def build_contract_vectorstore(force_rebuild: bool = False) -> Chroma:
    """Load contract markdown files, chunk them, and persist to Chroma."""
    if force_rebuild and VECTOR_DB_DIR.exists():
        import shutil
        shutil.rmtree(VECTOR_DB_DIR)

    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    if VECTOR_DB_DIR.exists() and any(VECTOR_DB_DIR.iterdir()) and not force_rebuild:
        print(f"Loading existing vector DB from: {VECTOR_DB_DIR}")
        return Chroma(persist_directory=str(VECTOR_DB_DIR), embedding_function=embeddings)

    print(f"Building vector DB from contracts in: {CONTRACTS_DIR}")
    docs = load_contract_documents()
    chunks = chunk_documents(docs, chunk_size=1200, chunk_overlap=200)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(VECTOR_DB_DIR),
    )

    print(f"Indexed {len(docs)} contracts into {len(chunks)} chunks.")
    return vectorstore


vectorstore = build_contract_vectorstore(force_rebuild=False)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
llm = ChatOpenAI(model_name=MODEL_NAME, temperature=0)

print("Contracts Q&A system is ready.")

In [ ]:
SYSTEM_PROMPT = """
You are a precise contracts assistant for Insurellm.
Answer only from the provided contract context and conversation history.
If the answer is not in context, clearly say you do not have enough information.
When possible, include short source citations using the contract filename.

Context:
{context}
""".strip()


def build_retrieval_query(question: str, history: list[dict]) -> str:
    """Use prior user turns to improve retrieval relevance."""
    prior_user_turns = [m["content"] for m in history if m.get("role") == "user"]
    if prior_user_turns:
        return "\n".join(prior_user_turns[-4:] + [question])
    return question


def answer_contract_question(question: str, history: list[dict] | None = None):
    history = history or []

    retrieval_query = build_retrieval_query(question, history)
    docs = retriever.invoke(retrieval_query)
    context = "\n\n".join(doc.page_content for doc in docs)

    messages = [SystemMessage(content=SYSTEM_PROMPT.format(context=context))]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    answer = response.content

    sources = sorted({doc.metadata.get("source", "unknown") for doc in docs})
    if sources:
        answer += "\n\nSources: " + ", ".join(sources)

    return answer, docs


# Quick test
sample_answer, sample_docs = answer_contract_question(
    "What are the termination notice requirements in these contracts?"
)
print(sample_answer)

In [ ]:
def chat_fn(message, history):
    """Gradio chat callback with history-aware retrieval."""
    answer, _ = answer_contract_question(message, history)
    return answer


demo = gr.ChatInterface(
    fn=chat_fn,
    title="Insurellm Contracts Q&A",
    description=(
        "Ask questions about company contracts in the knowledge base. "
        "Retrieval uses OpenAI embeddings + Chroma and includes chat history context."
    ),
    examples=[
        "Which contracts mention automatic renewal and what notice period is required?",
        "What overage fees are charged and for which products?",
        "Compare support response times across contracts.",
        "What are the data ownership terms?",
    ],
    type="messages",
)

# In notebooks, share=True is optional.
demo.launch(share=False)